# ⚖️ Re-Embedding Dataset Hukumonline dengan Multilingual E5-Large
**Mata Kuliah:** Natural Language Processing (UAS)

Notebook ini melakukan proses embedding terhadap 7.172 dokumen konsultasi hukum dari Klinik Hukumonline.
* **Model:** `intfloat/multilingual-e5-large` (Microsoft XLM-RoBERTa Large, 1024 dimensi)
* **Field Representasi:** `title + question + summarize`
* **Arsitektur E5:** Wajib menyertakan prefix `passage: ` untuk korpus dokumen
* **Output:** `vectors.npy` (matriks vektor) dan `documents.json` (metadata bersih)

## 1. Install & Import Dependensi

In [1]:
!pip install -q sentence-transformers torch tqdm numpy

In [1]:
import os
import json
import time
import zipfile
import urllib.request
import torch
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# Cek ketersediaan GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device aktif: {device.upper()}")
if device == "cuda":
    print(f"GPU Model   : {torch.cuda.get_device_name(0)}")
else:
    print("Perhatian: Berjalan di CPU. Jika menggunakan Colab, aktifkan GPU T4 di Runtime -> Change runtime type -> T4 GPU agar proses selesai dalam ~2-3 menit!")

Device aktif: CUDA
GPU Model   : Tesla T4


## 2. Memuat Dataset
Jika file belum ada di lingkungan Colab, notebook akan otomatis mengunduh `embedings.zip` dari repositori GitHub Anda.

In [2]:
# Otomatis unduh jika belum ada di container Colab
if not os.path.exists("embedings.json") and not os.path.exists("embedings.zip"):
    print("Dataset belum ada di container Colab. Mengunduh embedings.zip otomatis dari GitHub...")
    url = "https://github.com/sammdavid/UAS-Natural-Language-Processing/raw/main/embedings.zip"
    urllib.request.urlretrieve(url, "embedings.zip")
    print("Download selesai!")

raw_data = None
if os.path.exists("embedings.json"):
    print("Membaca langsung dari embedings.json...")
    with open("embedings.json", "r", encoding="utf-8") as f:
        raw_data = json.load(f)
elif os.path.exists("embedings.zip"):
    print("Membaca dari arsip embedings.zip...")
    with zipfile.ZipFile("embedings.zip", "r") as z:
        with z.open("embedings.json") as f:
            raw_data = json.load(f)
else:
    raise FileNotFoundError("File embedings.json atau embedings.zip tidak ditemukan!")

print(f"Total data berhasil dimuat: {len(raw_data)} entri")

Dataset belum ada di container Colab. Mengunduh embedings.zip otomatis dari GitHub...
Download selesai!
Membaca dari arsip embedings.zip...
Total data berhasil dimuat: 7172 entri


## 3. Ekstraksi Teks Representasi & Metadata
Sesuai rancangan arsitektur, teks yang di-embed adalah gabungan **`title + question + summarize`** dengan prefix `passage: `.
Field ulasan lengkap (`answer`) disimpan di metadata dokumen untuk digunakan sebagai konteks LLM.

In [3]:
corpus_passages = []
documents_metadata = []

for idx, item in enumerate(raw_data):
    title = item.get("title", "").strip()
    question = item.get("question", "").strip()
    summarize = item.get("summarize", "").strip()
    
    # Format wajib E5 untuk dokumen
    passage_text = f"passage: {title}. {question} {summarize}".strip()
    corpus_passages.append(passage_text)
    
    # Metadata bersih
    documents_metadata.append({
        "id": idx,
        "title": title,
        "question": question,
        "summarize": summarize,
        "answer": item.get("answer", ""),
        "source": item.get("source", "")
    })

print(f"Total passages siap di-encode: {len(corpus_passages)}")
print("\nContoh passage pertama:")
print(corpus_passages[0][:300], "...")

Total passages siap di-encode: 7172

Contoh passage pertama:
passage: Kiat-kiat Membeli Tanah Girik yang Aman. Bagaimana cara pengurusan pembelian tanah girik (tanpa sertifikat hak milik) sesuai peraturan? Girik bukanlah tanda bukti atas tanah, tetapi bukti bahwa pemilik girik menguasai tanah milik adat dan sebagai pembayar pajak atas bidang tanah tersebut be ...


## 4. Memuat Model Multilingual E5-Large
Model resmi Microsoft (`intfloat/multilingual-e5-large`) dengan dimensi vektor **1024**.

In [4]:
model_name = "intfloat/multilingual-e5-large"
print(f"Mengunduh dan memuat model: {model_name}...")
model = SentenceTransformer(model_name, device=device)
print("Model berhasil dimuat!")

Mengunduh dan memuat model: intfloat/multilingual-e5-large...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Model berhasil dimuat!


## 5. Komputasi Vektor Embedding
Menghasilkan representasi vektor float32 (L2-normalized) untuk seluruh 7.172 dokumen.

In [5]:
batch_size = 32 if device == "cuda" else 16
print(f"Memulai embedding dengan batch_size={batch_size}...")
start_time = time.time()

vectors = model.encode(
    corpus_passages,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

elapsed = time.time() - start_time
print(f"Selesai dalam {elapsed:.2f} detik! ({elapsed/60:.2f} menit)")
print(f"Bentuk matriks vektor: {vectors.shape} (Jumlah dokumen x Dimensi)")

Memulai embedding dengan batch_size=32...


Batches:   0%|          | 0/225 [00:00<?, ?it/s]

Selesai dalam 604.72 detik! (10.08 menit)
Bentuk matriks vektor: (7172, 1024) (Jumlah dokumen x Dimensi)


## 6. Simpan Hasil (`vectors.npy` & `documents.json`)

In [6]:
# 1. Simpan matriks vektor (format numpy float32, ukuran ~29 MB)
np.save("vectors.npy", vectors.astype(np.float32))

# 2. Simpan metadata dokumen (format json, ukuran ~15-20 MB)
with open("documents.json", "w", encoding="utf-8") as f:
    json.dump(documents_metadata, f, ensure_ascii=False, indent=2)

print("File berhasil disimpan:")
print(f"- vectors.npy    : {os.path.getsize('vectors.npy') / (1024*1024):.2f} MB")
print(f"- documents.json : {os.path.getsize('documents.json') / (1024*1024):.2f} MB")

File berhasil disimpan:
- vectors.npy    : 28.02 MB
- documents.json : 57.25 MB


## 7. Sanity Check / Uji Coba Pencarian Semantik Sederhana
Menguji vektor baru dengan pertanyaan contoh untuk memverifikasi akurasi retrieval.

In [7]:
def test_search(query_text, top_k=3):
    # Format wajib E5 untuk pertanyaan
    formatted_q = f"query: {query_text.strip()}"
    q_vec = model.encode(formatted_q, normalize_embeddings=True, convert_to_numpy=True)
    
    # Cosine similarity via dot product
    scores = np.dot(vectors, q_vec)
    top_ids = np.argsort(scores)[::-1][:top_k]
    
    print(f"=== Hasil Pencarian: '{query_text}' ===\n")
    for rank, idx in enumerate(top_ids, 1):
        doc = documents_metadata[idx]
        print(f"[{rank}] Skor: {scores[idx]:.4f}")
        print(f"     Judul: {doc['title']}")
        print(f"     Kasus: {doc['question'][:120]}...")
        print(f"     URL  : {doc['source']}\n")

# Uji coba dengan studi kasus hukum nyata
test_search("bagaimana cara membeli tanah girik tanpa sertifikat yang aman?")
test_search("provisions for foreign chef working in indonesia")

=== Hasil Pencarian: 'bagaimana cara membeli tanah girik tanpa sertifikat yang aman?' ===

[1] Skor: 0.8989
     Judul: Kiat-kiat Membeli Tanah Girik yang Aman
     Kasus: Bagaimana cara pengurusan pembelian tanah girik (tanpa sertifikat hak milik) sesuai peraturan?...
     URL  : https://www.hukumonline.com/klinik/a/kiat-kiat-membeli-tanah-girik-yang-aman-cl1360/

[2] Skor: 0.8689
     Judul: Tanah Warisan Belum Bersertifikat, Begini Cara Mengurusnya
     Kasus: Kami memiliki sebidang tanah yang berasal dari tanah warisan. Namun, tanah tersebut belum mempunyai surat-surat kepemili...
     URL  : https://www.hukumonline.com/klinik/a/tanah-warisan-belum-bersertifikat--begini-cara-mengurusnya-lt521ec8e766f1f/

[3] Skor: 0.8649
     Judul: Sahkah Jual Beli Tanah antara Ayah dengan Anak?
     Kasus: Apakah sah secara hukum jual beli tanah antara ayah dengan anak? Terima kasih....
     URL  : https://www.hukumonline.com/klinik/a/jual-beli-lt4f8ad753797a8/

=== Hasil Pencarian: 'provisions f

## 8. Download Hasil (Jika Menggunakan Google Colab)
Jalankan cell ini jika Anda menjalankan notebook di Google Colab dan ingin mengunduh kedua file hasil ke komputer lokal Anda.

In [8]:
try:
    from google.colab import files
    print("Mengunduh vectors.npy...")
    files.download("vectors.npy")
    print("Mengunduh documents.json...")
    files.download("documents.json")
except ImportError:
    print("Bukan di Google Colab. File sudah tersimpan di direktori lokal Anda.")

Mengunduh vectors.npy...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Mengunduh documents.json...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. SANITY CHECK 2
Pastikan file vectors.npy dan documents.json  sudah didalam folder

In [10]:
import numpy as np
import json

# 1. Load data
vectors = np.load("vectors.npy")
with open("documents.json", "r", encoding="utf-8") as f:
    docs = json.load(f)

# 2. Check shapes & counts
print("Vectors shape :", vectors.shape)       # MUST BE: (7172, 1024)
print("Docs count    :", len(docs))             # MUST BE: 7172
print("Has NaN/Inf   :", np.isnan(vectors).any() or np.isinf(vectors).any()) # MUST BE: False

# 3. Check L2 Norms (should be 1.0 because normalize_embeddings=True)
norms = np.linalg.norm(vectors, axis=1)
print(f"Norm range    : {norms.min():.4f} to {norms.max():.4f}") # MUST BE: ~1.0000

Vectors shape : (7172, 1024)
Docs count    : 7172
Has NaN/Inf   : False
Norm range    : 1.0000 to 1.0000


In [11]:
import numpy as np
import json
import random
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-large")
vectors = np.load("vectors.npy")
with open("documents.json", "r", encoding="utf-8") as f:
    docs = json.load(f)

# Sample 50 random documents as benchmark queries
random.seed(42)
test_indices = random.sample(range(len(docs)), 50)

hits_at_1 = 0
hits_at_5 = 0
reciprocal_ranks = []

print("Running automated benchmark on 50 sample queries...\n")
for true_idx in test_indices:
    sample_q = docs[true_idx]["question"].strip()
    
    # Encode with E5 query prefix
    q_vec = model.encode(f"query: {sample_q}", normalize_embeddings=True)
    
    # Dot product similarity
    scores = np.dot(vectors, q_vec)
    top_indices = np.argsort(scores)[::-1][:5]
    
    # Check if true document is in top results
    if true_idx == top_indices[0]:
        hits_at_1 += 1
    if true_idx in top_indices:
        hits_at_5 += 1
        rank = list(top_indices).index(true_idx) + 1
        reciprocal_ranks.append(1.0 / rank)
    else:
        reciprocal_ranks.append(0.0)

print(f"=== BENCHMARK RESULTS (Evaluation) ===")
print(f"Hit@1 (Top-1 Accuracy) : {hits_at_1 / len(test_indices) * 100:.2f}%  (Target: >85%)")
print(f"Hit@5 (Top-5 Accuracy) : {hits_at_5 / len(test_indices) * 100:.2f}%  (Target: >95%)")
print(f"MRR   (Mean Recip Rank): {np.mean(reciprocal_ranks):.4f}      (Target: >0.90)")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Running automated benchmark on 50 sample queries...

=== BENCHMARK RESULTS (Evaluation) ===
Hit@1 (Top-1 Accuracy) : 100.00%  (Target: >85%)
Hit@5 (Top-5 Accuracy) : 100.00%  (Target: >95%)
MRR   (Mean Recip Rank): 1.0000      (Target: >0.90)


In [12]:
# Tes 1: Bahasa Gaul / Slang Ketenagakerjaan
test_search("bos gamau cairin uang lembur lapor kemana ya?")

# Tes 2: Kasus Sengketa Tanah Tanpa Sebut Kata 'Girik'
test_search("beli tanah adat yang belum ada sertifikat dari bpn gimana caranya biar ga diserobot orang?")

# Tes 3: Kasus Pidana Sehari-hari
test_search("duit kantor kepake sama temen tapi udah dicicil balikin apa tetep dipenjara?")

# Tes 4: Bahasa Inggris Kasus Hukum Indonesia (Cross-Lingual)
test_search("how can a foreigner legally own property or land in bali?")

=== Hasil Pencarian: 'bos gamau cairin uang lembur lapor kemana ya?' ===

[1] Skor: 0.8067
     Judul: Hukumnya Meminta Uang Tebusan Binatang Piaraan Milik Orang Lain
     Kasus: Gan, ane mohon tanya. Sebelumnya ane minta maaf kalau sekiranya pertanyaan ane kurang berbobot atau tidak masuk kategori...
     URL  : https://www.hukumonline.com/klinik/a/hukumnya-meminta-uang-tebusan-binatang-piaraan-milik-orang-lain-lt4f76a86381246/

[2] Skor: 0.8054
     Judul: Kapan Seharusnya Uang Pisah Cair?
     Kasus: Kapankah idealnya uang pisah/UPH/UPMK dibayarkan oleh perusahaan saat kita mengundurkan diri dari perusahaan tersebut? A...
     URL  : https://www.hukumonline.com/klinik/a/kapan-seharusnya-uang-pisah-cair-lt559f647a75eed/

[3] Skor: 0.8050
     Judul: Karyawan Wajib Share Location saat Liburan, Langgar Privasi?
     Kasus: Halo. Saya mau nanya, ada perusahaan yang mewajibkan karyawan untuk share location saat libur kerja. Yang artinya ruang ...
     URL  : https://www.hukumonline.com/k